In [4]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [5]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))  # MNIST标准化参数
])


In [6]:
train_data = datasets.MNIST(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_data = datasets.MNIST(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

In [7]:
batch_size = 64  # 可调整的超参数
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

In [8]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3),  # 输入通道1（灰度图）
            nn.ReLU(),
            nn.MaxPool2d(2),                 # 输出尺寸：14x14
            nn.Conv2d(32, 64, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2)                  # 输出尺寸：5x5
        )
        self.fc_layers = nn.Sequential(
            nn.Linear(64 * 5 * 5, 128),
            nn.ReLU(),
            nn.Linear(128, 10)                # 输出10个类别（0-9）
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)  # 展平为向量
        x = self.fc_layers(x)
        return x

In [9]:
model = CNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [10]:
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    for images, labels in train_loader:  # 使用已定义的train_loader
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    # 验证精度
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    print(f"Epoch {epoch+1}/{num_epochs}, Accuracy: {100 * correct / total:.2f}%")

Epoch 1/10, Accuracy: 98.10%
Epoch 2/10, Accuracy: 98.82%
Epoch 3/10, Accuracy: 98.74%
Epoch 4/10, Accuracy: 99.18%
Epoch 5/10, Accuracy: 99.19%
Epoch 6/10, Accuracy: 98.86%
Epoch 7/10, Accuracy: 98.96%
Epoch 8/10, Accuracy: 99.13%
Epoch 9/10, Accuracy: 99.16%
Epoch 10/10, Accuracy: 99.04%
